# ScanDar on Colab

Training moves between a local RTX 3060 and Colab depending on what is available, so nothing in the codebase assumes either one. This notebook makes a Colab runtime look like the local machine:

1. check what GPU we were given
2. mount Drive, so **checkpoints survive a session timeout**
3. clone the repo and install it
4. copy the data onto the runtime's own disk and point `SCANDAR_DATA` / `SCANDAR_OUT` where they belong
5. run the sanity checks

After that, every other notebook and every `python train.py ...` command runs unchanged.

> **One-time setup:** copy `data/` to `MyDrive/scandar/data`. The source images are small (~75 MB with the backgrounds and the real photos); the six frozen evaluation buckets add ~400 MB on top. **Copy them rather than regenerating them** — regenerating is possible (step 5) but the buckets are frozen from two different configs, so the wrong command produces buckets that are not the ones every number in this project was measured against.

In [ ]:
# 1. what did we get?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Drive — checkpoints and figures live here so a timeout costs nothing
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/scandar'

In [ ]:
# 3. code
REPO_URL = 'https://github.com/SepehrGhr/ScanDar.git'

import os
if not os.path.isdir('/content/ScanDar'):
    !git clone $REPO_URL /content/ScanDar
%cd /content/ScanDar
!git pull --ff-only

# Colab already ships torch + CUDA, so requirements.txt deliberately does not pin them.
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# 4. data on the runtime's own disk, outputs on Drive
#
# Two different jobs, so two different places. The generator opens scans and
# background photos inside every __getitem__ across eight worker processes, and
# Drive is a network filesystem — reading the training data through it is slow in
# exactly the place this project is already bottlenecked. Checkpoints are the
# opposite: written once an epoch, and worthless if the session dies with them on
# a local disk.
import os, shutil, subprocess

LOCAL_DATA = '/content/data'
if not os.path.isdir(LOCAL_DATA):
    print('copying data from Drive (a few minutes, once per session)...')
    shutil.copytree(f'{DRIVE_ROOT}/data', LOCAL_DATA)
print(subprocess.run(['du', '-sh', LOCAL_DATA], capture_output=True, text=True).stdout)

os.environ['SCANDAR_DATA'] = LOCAL_DATA
os.environ['SCANDAR_OUT'] = f'{DRIVE_ROOT}/outputs'

import scandar
print('data   ->', scandar.paths.data)
print('output ->', scandar.paths.out)

In [ ]:
# 5. verify before spending GPU time on a broken setup
!python scripts/prepare_data.py
!python scripts/sanity_checks.py

# If — and only if — the frozen buckets did not come across in the copy above,
# regenerate them **per task, from the config each was frozen with**. The default
# config produces enhancement buckets on the wrong canvas, and a wrong bucket is
# worse than a missing one because nothing downstream complains. Both commands
# are no-ops when the sets are already there.
#
# !python scripts/freeze_eval_sets.py --config configs/enhance_realistic.yaml --task enhance
# !python scripts/freeze_eval_sets.py --config configs/corner.yaml           --task corner

## Training here

```bash
python train.py --config configs/enhance.yaml
```

Every run checkpoints into `SCANDAR_OUT` with its optimiser, scaler and RNG state, so a session that dies mid-epoch resumes with `train.resume=auto` (the default) — **re-running the exact same command is the way to resume**. `train.max_hours` stops a run cleanly before Colab stops it messily:

```bash
python train.py --config configs/enhance.yaml --set train.max_hours=3
```

**Expect Colab to be the slower machine for this project**, which inverts the usual advice. The bottleneck is not the GPU — it is the CPU compositing and degrading synthetic photos, and a Colab runtime has about two cores against the development laptop's sixteen. Watch the `samples/s` figure the trainer prints at the end of the first epoch and multiply it out before committing to a run; the laptop manages about 18 on the enhancement task and about 6 on the corner task.

Because of that, a bigger batch will not buy throughput here. `batch_size` and `grad_accum` are separate keys so that the *effective* batch can be held constant on a machine where the real one does not fit — at 256x256 patches the default batch of 16 peaks around 3 GB, so it fits everywhere and neither key needs changing. They exist for the day a model or a resolution does not.

**Run one thing at a time.** Two trainings in one runtime share the same two cores and both slow down; the wall clock is the same and the failure modes are worse.

## The regularisation study, one arm at a time

The comparison only means something if every arm matches the baseline it is scored against — same schedule, same seed, same data — so the arms are run exactly as written, with no extra `--set` beyond the wall-clock guard. The baselines (`enhance_realistic`, `corner_reg`, `corner_heat`) are already trained and do not need re-running.

Run the cell below once per arm by editing `ARM`. When a session dies, re-run the identical cell: it resumes from `last.pt` on Drive. `docs/dropout-study.md` explains what each arm is and what it is expected to show; `docs/running-on-colab.md` has the full session-by-session recipe.

In [ ]:
# one arm of the regularisation study — edit ARM, re-run to resume
ARM = 'corner_reg_dropout'   # corner_heat_dropout | enhance_dropout | *_dropout_wide
HOURS = 3.0                  # stop cleanly before Colab does; resume by re-running

!python train.py --config configs/{ARM}.yaml --set train.max_hours={HOURS}

In [ ]:
# when an arm has finished all its epochs, score it and refresh the tables
!python evaluate.py --config configs/{ARM}.yaml
!python scripts/compare_dropout.py
!python scripts/make_figures.py --run {ARM}